# 7장. RAG 시스템 평가 — 03. LangSmith 데이터셋과 LLM-as-Judge 평가

## 학습 목표

- **LangSmith(랭스미스)**에 평가 데이터셋을 업로드하고 관리하는 방법 이해
- **LLM-as-Judge** 패턴으로 자동 평가 구현
- 임베딩(Embedding) 유사도 기반 평가 방법 적용

## 사전 지식

- LangSmith 계정 및 API 키 설정 (`.env`에 `LANGCHAIN_API_KEY`)
- RAGAS 평가 경험 (02번 노트북)

## LangSmith 평가 워크플로우

```mermaid
flowchart LR
    A[📊 평가 데이터셋<br/>inputs + outputs] -->|create_dataset| B[LangSmith<br/>클라우드 저장]
    B --> C[evaluate 함수<br/>실행]
    D[🤖 RAG 시스템<br/>평가 대상] --> C
    E[⚖️ 평가자<br/>Evaluator] --> C
    C --> F[실험 결과<br/>Experiment]
    F --> G[📈 LangSmith<br/>대시보드]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A input
    class D,E process
    class B,F storage
    class C process
    class G output
```

---

## 환경 설정

In [ ]:
# 필요한 패키지 설치
# !pip install -qU langsmith langchain langchain-openai


In [ ]:
from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# LangSmith 설정 확인
print(f"LANGCHAIN_API_KEY: {'✅ 설정됨' if os.getenv('LANGCHAIN_API_KEY') else '❌ 미설정'}")
print(f"LANGCHAIN_TRACING_V2: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")
print(f"LANGCHAIN_PROJECT: {os.getenv('LANGCHAIN_PROJECT', '기본값')}")


---

## LangSmith 데이터셋 생성

LangSmith에서 데이터셋은 `inputs`와 `outputs`(선택)로 구성돼요.

- **inputs**: 시스템에 전달할 입력 (예: `{"question": "..."}`)
- **outputs**: 기대하는 정답 (예: `{"answer": "..."}`) — 평가자가 비교 기준으로 사용

> **실무 팁**: 동일한 데이터셋을 여러 실험(experiment)에서 재사용할 수 있어요. 모델이나 청크 크기를 바꾸면서 같은 데이터셋으로 비교하는 것이 좋은 관행입니다.

In [ ]:
# ---------------------------------------------------
# 평가용 질문-답변 쌍 준비 및 데이터프레임 생성
# ---------------------------------------------------
import pandas as pd
from langsmith import Client

# LangSmith 클라이언트 초기화 — LANGCHAIN_API_KEY 환경변수를 자동으로 읽음
client = Client()

# 평가용 질문과 답변 준비
inputs = [
    "What is the Transformer architecture?",
    "How does self-attention mechanism work?",
    "What are the advantages of Transformers over RNNs?",
    "Explain the encoder-decoder structure in Transformers.",
    "What is multi-head attention?",
]

outputs = [
    "The Transformer is a neural network architecture based entirely on attention mechanisms, without recurrent or convolutional layers.",
    "Self-attention computes attention scores between all positions in a sequence, allowing the model to weigh the importance of different parts.",
    "Transformers can process sequences in parallel, capture long-range dependencies better, and are more efficient to train than RNNs.",
    "The Transformer consists of an encoder that processes the input and a decoder that generates the output, both using multi-head attention.",
    "Multi-head attention applies multiple attention mechanisms in parallel, allowing the model to focus on different aspects of the information.",
]

# 데이터프레임 생성
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

print(f"평가 데이터: {len(df)}개")
df.head()

### LangSmith에 데이터셋 업로드

`read_dataset`으로 기존 데이터셋을 확인하고, 없으면 `create_dataset`으로 새로 만들어요. 동일한 이름의 데이터셋이 중복 생성되지 않도록 예외 처리를 합니다.

In [ ]:
# ---------------------------------------------------
# LangSmith에 데이터셋 업로드 (중복 생성 방지 처리 포함)
# ---------------------------------------------------
# ============================================================
# TODO: 데이터셋이 없으면 create_dataset()과 create_example()로 생성하세요
# 힌트: try: client.read_dataset(...) except LangSmithNotFoundError: client.create_dataset(...)
# 예상 결과: "기존 데이터셋 사용" 또는 "새 데이터셋 생성" 출력
# ============================================================

from langsmith import utils as ls_utils

# 데이터셋 이름
dataset_name = "transformer-qa-evaluation"



---

## 평가할 RAG 시스템 구축

In [ ]:
import os

# 데이터 파일 경로 확인
file_path = "data/attention_paper.pdf"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}\n현재 디렉토리: {os.getcwd()}")

In [ ]:
# ---------------------------------------------------
# RAG 시스템 구축 (벡터 DB + 체인)
# ---------------------------------------------------
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계: 문서 로드 및 분할
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

# 2단계: 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """Answer the question based on the context provided.

Context: {context}

Question: {question}

Answer:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 시스템 구축 완료")

### RAG 시스템 동작 확인

In [ ]:
# 테스트 질문
test_q = "What is the Transformer architecture?"
answer = rag_chain.invoke(test_q)

print("질문:", test_q)
print("\n답변:")
print(answer)


---

## LLM-as-Judge 평가

**LLM-as-Judge**는 LLM을 평가자로 활용해 답변 품질을 자동으로 판단하는 방식이에요. 사람이 평가하는 것과 유사한 정성적 판단을 자동화할 수 있어요.

LangSmith는 다양한 사전 정의된 평가자를 제공해요:

| 평가자 | 설명 |
|--------|------|
| `labeled_criteria` | 참조 답변과 비교하여 기준 충족 여부 평가 |
| `criteria` | 참조 없이 특정 기준으로 평가 |
| `embedding_distance` | 임베딩 벡터 거리로 유사도 평가 |

> **실무 팁**: LLM-as-Judge는 편향이 생길 수 있어요. 평가 LLM과 생성 LLM이 같은 경우 자기 자신의 답변에 높은 점수를 주는 경향이 있습니다. 가능하면 더 강력한 모델(GPT-4 계열)을 평가자로 사용하세요.

In [ ]:
# ---------------------------------------------------
# LLM-as-Judge 평가자 설정
# ---------------------------------------------------
# ============================================================
# TODO: LangChainStringEvaluator("labeled_criteria", ...)로 평가자를 만들고 evaluate()를 실행하세요
# 힌트: prepare_data=lambda run, example: {"prediction": run.outputs["output"], "reference": example.outputs["answer"], ...}
# 예상 결과: experiment_results.experiment_url 출력
# ============================================================

from langsmith.evaluation import LangChainStringEvaluator, evaluate



### 평가 실행

In [ ]:
# ---------------------------------------------------
# LangSmith evaluate() 실행 — 결과는 대시보드에서 확인 가능
# ---------------------------------------------------
# ============================================================
# TODO: evaluate()를 호출하여 LLM-as-Judge 평가를 실행하세요
# 힌트: evaluate(rag_qa_function, data=dataset_name, evaluators=[criteria_evaluator], ...)
# 예상 결과: experiment_results.experiment_url 출력
# ============================================================



---

## 임베딩 유사도 평가

**임베딩 거리** 기반 평가는 참조 답변과 생성 답변을 벡터로 변환한 뒤 코사인 거리를 측정해요. LLM 호출 없이 의미적 유사도를 빠르게 측정할 수 있습니다.

In [ ]:
# ---------------------------------------------------
# 임베딩 거리 기반 평가자 설정 및 실행
# ---------------------------------------------------
# ============================================================
# TODO: "embedding_distance" 평가자를 만들고 evaluate()를 실행하세요
# 힌트: LangChainStringEvaluator("embedding_distance", config={"embeddings": embeddings, "distance_metric": EmbeddingDistance.COSINE})
# 예상 결과: embedding_results.experiment_url 출력
# ============================================================

from langchain_community.evaluation import EmbeddingDistance
from langsmith.evaluation import evaluate



---

## 정리

### LangSmith 평가 핵심 흐름

1. `Client()`로 LangSmith에 연결
2. `create_dataset()` + `create_example()`로 데이터셋 업로드
3. 평가할 시스템을 `inputs -> outputs` 함수로 래핑
4. `LangChainStringEvaluator`로 평가자 정의
5. `evaluate()`로 실험 실행 → LangSmith 대시보드에서 확인

### RAGAS vs LangSmith

| 특징 | RAGAS | LangSmith |
|------|-------|-----------|
| 초점 | RAG 특화 4가지 지표 | 범용 평가 + 프로덕션 모니터링 |
| 평가 방식 | RAG 메트릭 | LLM-as-Judge, 커스텀 |
| 데이터셋 | 로컬 관리 | 클라우드 관리 |
| 시각화 | Python 코드 | 웹 대시보드 |
| 추적 | 없음 | 실시간 트레이싱 |

### LLM-as-Judge 장단점

**장점**: 자동화, 확장 가능, 일관된 기준 적용<br/>
**단점**: 평가 LLM 편향 가능성, 추가 API 비용, 복잡한 케이스 오판 가능

### 다음 노트북 예고

LangSmith가 제공하는 기본 평가자 외에 **직접 커스텀 평가 로직**을 작성하는 방법을 배워볼게요. 규칙 기반, LLM 기반, 임베딩 기반 세 가지 유형을 직접 구현합니다.